### Loading Pretrained BERT and ViT Models

In this step, I load the pretrained BERT and Vision Transformer (ViT) models. BERT will be used to extract text features from the questions and answer options, while ViT will extract image features. Both models will remain frozen initially to reduce training time and computational cost.

In [13]:
import torch
import torch.nn as nn
import numpy as np

from transformers import AutoModel
from datasets import load_from_disk

# Select device
device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

print("Using device:", device)

# Load preprocessed dataset
processed_dataset = load_from_disk(
    "../data/processed/scienceqa_preprocessed"
)

# Model names
BERT_MODEL = "bert-base-uncased"
VIT_MODEL = "google/vit-base-patch16-224"

# Load pretrained models
bert = AutoModel.from_pretrained(BERT_MODEL)
vit = AutoModel.from_pretrained(VIT_MODEL)

# Freeze both models
for param in bert.parameters():
    param.requires_grad = False

for param in vit.parameters():
    param.requires_grad = False

bert = bert.to(device)
vit = vit.to(device)

bert.eval()
vit.eval()

print("BERT and ViT loaded successfully.")

Using device: mps


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT and ViT loaded successfully.


### Fixing MPS Attention Compatibility

The default attention implementation caused a compatibility issue with Apple's MPS device. I changed the attention implementation to eager attention to avoid this issue while keeping the pretrained models frozen.

In [14]:
# Use eager attention instead of SDPA
bert = AutoModel.from_pretrained(
    BERT_MODEL,
    attn_implementation="eager"
)

vit = AutoModel.from_pretrained(
    VIT_MODEL,
    attn_implementation="eager"
)

# Freeze both models
for param in bert.parameters():
    param.requires_grad = False

for param in vit.parameters():
    param.requires_grad = False

bert = bert.to(device)
vit = vit.to(device)

bert.eval()
vit.eval()

print("BERT and ViT reloaded with eager attention.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT and ViT reloaded with eager attention.


### Checking Feature Dimensions

This step checks the output dimensions of BERT and ViT using one sample from the dataset. I will use these features later to combine the text and image information in the early fusion model.

In [15]:
sample = processed_dataset["train"][0]

# Prepare text inputs
input_ids = torch.tensor(
    sample["input_ids"], dtype=torch.long
).unsqueeze(0).to(device)

attention_mask = torch.tensor(
    sample["attention_mask"], dtype=torch.long
).unsqueeze(0).to(device)

# Prepare image input
pixel_values = torch.tensor(
    sample["pixel_values"], dtype=torch.float32
).unsqueeze(0).to(device)

# Extract features
with torch.no_grad():

    # BERT features for each answer option
    batch_size, num_choices, seq_length = input_ids.shape

    bert_output = bert(
        input_ids=input_ids.view(-1, seq_length),
        attention_mask=attention_mask.view(-1, seq_length)
    )

    text_features = bert_output.last_hidden_state[:, 0, :]

    # ViT image features
    vit_output = vit(pixel_values=pixel_values)

    image_features = vit_output.last_hidden_state[:, 0, :]

print("Text feature shape:", text_features.shape)
print("Image feature shape:", image_features.shape)

Text feature shape: torch.Size([5, 768])
Image feature shape: torch.Size([1, 768])


### Building the Early Fusion Model

In this step, I combine the text features extracted by BERT and the image features extracted by ViT. The two feature representations are concatenated and passed through a classification layer to score each answer option. Both pretrained models remain frozen, while the fusion layers are trained.

In [16]:
class EarlyFusionModel(nn.Module):
    def __init__(self, bert, vit):
        super().__init__()

        self.bert = bert
        self.vit = vit

        # Freeze pretrained models
        for param in self.bert.parameters():
            param.requires_grad = False

        for param in self.vit.parameters():
            param.requires_grad = False

        # Fusion classifier
        self.fusion = nn.Sequential(
            nn.Linear(768 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def forward(self, input_ids, attention_mask, pixel_values, option_mask):

        batch_size, num_choices, seq_length = input_ids.shape

        input_ids = input_ids.view(-1, seq_length)
        attention_mask = attention_mask.view(-1, seq_length)

        valid_indices = option_mask.view(-1).nonzero(as_tuple=True)[0]

        # Extract text features
        with torch.no_grad():
            text_output = self.bert(
                input_ids=input_ids[valid_indices],
                attention_mask=attention_mask[valid_indices]
            )

            text_features = text_output.last_hidden_state[:, 0, :]

            # Extract image features
            image_output = self.vit(pixel_values=pixel_values)
            image_features = image_output.last_hidden_state[:, 0, :]

        # Repeat image features for each valid answer option
        image_features = image_features.repeat_interleave(
            num_choices, dim=0
        )[valid_indices]

        # Concatenate text and image features
        combined_features = torch.cat(
            (text_features, image_features), dim=1
        )

        # Calculate answer scores
        valid_scores = self.fusion(combined_features).squeeze(-1)

        scores = torch.full(
            (batch_size * num_choices,),
            float("-inf"),
            device=input_ids.device
        )

        scores[valid_indices] = valid_scores

        return scores.view(batch_size, num_choices)


early_fusion_model = EarlyFusionModel(bert, vit).to(device)

print("Early Fusion model created successfully.")

Early Fusion model created successfully.


### Preparing DataLoaders for Early Fusion

This step prepares the training, validation and test datasets for the early fusion model. Each batch contains the question, answer options, and corresponding images so that BERT and ViT can process both modalities together.

In [17]:
from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk

# Load preprocessed dataset
processed_dataset = load_from_disk(
    "../data/processed/scienceqa_preprocessed"
)

class EarlyFusionDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        sample = self.dataset[idx]

        return {
            "input_ids": torch.tensor(sample["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(sample["attention_mask"], dtype=torch.long),
            "option_mask": torch.tensor(sample["option_mask"], dtype=torch.bool),
            "pixel_values": torch.tensor(sample["pixel_values"], dtype=torch.float32),
            "labels": torch.tensor(sample["labels"], dtype=torch.long)
        }

# Create datasets
train_dataset = EarlyFusionDataset(processed_dataset["train"])
val_dataset = EarlyFusionDataset(processed_dataset["validation"])
test_dataset = EarlyFusionDataset(processed_dataset["test"])

# Create DataLoaders
BATCH_SIZE = 8

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 442
Validation batches: 168
Test batches: 253


Model testing

In [18]:
batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
option_mask = batch["option_mask"].to(device)
pixel_values = batch["pixel_values"].to(device)

with torch.no_grad():
    logits = early_fusion_model(
        input_ids,
        attention_mask,
        pixel_values,
        option_mask
    )

print("Input text shape:", input_ids.shape)
print("Input image shape:", pixel_values.shape)
print("Output shape:", logits.shape)
print("Sample logits:", logits[0])

Input text shape: torch.Size([8, 5, 128])
Input image shape: torch.Size([8, 3, 224, 224])
Output shape: torch.Size([8, 5])
Sample logits: tensor([-0.5625, -0.4465, -0.4099, -0.2841,    -inf], device='mps:0')


### Setting Up Early Fusion Training

This cell defines the cross-entropy loss function and AdamW optimizer. Only the fusion classifier will be trained, while the pretrained BERT and ViT models remain frozen.

In [19]:
early_fusion_model = EarlyFusionModel(bert, vit).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    early_fusion_model.fusion.parameters(),
    lr=1e-3,
    weight_decay=0.01
)

print("Early Fusion model and optimizer recreated.")

Early Fusion model and optimizer recreated.


### Training and Validation Functions

These functions train the early fusion model and evaluate its performance using the validation dataset. During training, the fusion classifier learns to combine image and text features to predict the correct answer. BERT and ViT remain frozen.

In [20]:
from tqdm.auto import tqdm
import copy

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Training", leave=False)

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        option_mask = batch["option_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        logits = model(
            input_ids,
            attention_mask,
            pixel_values,
            option_mask
        )

        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / len(loader), correct / total


def evaluate(model, loader, criterion):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Validation", leave=False)

    with torch.no_grad():
        for batch in progress_bar:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            option_mask = batch["option_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            labels = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask,
                pixel_values,
                option_mask
            )

            loss = criterion(logits, labels)

            total_loss += loss.item()

            predictions = logits.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

### Training the Early Fusion Model

The model is trained for a maximum of 20 epochs with early stopping patience of 4. After each epoch, training and validation loss and accuracy are recorded. The model weights with the lowest validation loss are saved and restored after training.

In [21]:
EPOCHS = 20
PATIENCE = 4

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
best_model_state = None

history_early = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch + 1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        early_fusion_model,
        train_loader,
        optimizer,
        criterion
    )

    val_loss, val_acc = evaluate(
        early_fusion_model,
        val_loader,
        criterion
    )

    history_early["train_loss"].append(train_loss)
    history_early["train_accuracy"].append(train_acc)
    history_early["val_loss"].append(val_loss)
    history_early["val_accuracy"].append(val_acc)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        best_model_state = copy.deepcopy(
            early_fusion_model.state_dict()
        )
    else:
        epochs_without_improvement += 1

    print(
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Accuracy: {val_acc:.4f}"
    )

    print(
        f"Best Val Loss: {best_val_loss:.4f} "
        f"(Epoch {best_epoch})"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch + 1}.")
        break

early_fusion_model.load_state_dict(best_model_state)

print("\nTraining completed.")
print("Best epoch:", best_epoch)
print("Best validation loss:", round(best_val_loss, 4))


Epoch 1/20


Training:   0%|          | 0/442 [00:00<?, ?it/s]

Validation:   0%|          | 0/168 [00:00<?, ?it/s]

Train Loss: 1.2238 | Train Accuracy: 0.3317 | Val Loss: 1.1347 | Val Accuracy: 0.4795
Best Val Loss: 1.1347 (Epoch 1)

Epoch 2/20


Training:   0%|          | 0/442 [00:00<?, ?it/s]

Validation:   0%|          | 0/168 [00:00<?, ?it/s]

... (51 outputs left out)

Train Loss: 1.1248 | Train Accuracy: 0.4031 | Val Loss: 1.0127 | Val Accuracy: 0.5168
Best Val Loss: 1.0127 (Epoch 19)

Epoch 20/20


Training:   0%|          | 0/442 [00:00<?, ?it/s]

Validation:   0%|          | 0/168 [00:00<?, ?it/s]

Train Loss: 1.1061 | Train Accuracy: 0.4224 | Val Loss: 1.0423 | Val Accuracy: 0.5197
Best Val Loss: 1.0127 (Epoch 19)

Training completed.
Best epoch: 19
Best validation loss: 1.0127


In [22]:
test_loss, test_accuracy = evaluate(
    early_fusion_model,
    test_loader,
    criterion
)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Validation:   0%|          | 0/253 [00:00<?, ?it/s]

Test Loss: 0.9830
Test Accuracy: 0.4963


### Experiment 1 Early Fusion Test Results

The Early Fusion model achieved a test accuracy of 49.63% and a test loss of 0.9830. Compared with the text-only BERT baseline, its test accuracy was lower by 7.14 percentage points.

This indicates that simple concatenation of frozen BERT and ViT features did not improve performance. One possible reason is that the frozen image and text representations were not sufficiently adapted to the ScienceQA task.

## Experiment 2: Early Fusion V2 – Fine-tuned BERT

In this experiment, I will improve the previous early fusion model by fine-tuning the last four layers of BERT while keeping ViT frozen. The aim is to allow BERT to better understand the ScienceQA questions while retaining the pretrained image features.

In [23]:
# Reload pretrained models
bert_v2 = AutoModel.from_pretrained(
    BERT_MODEL,
    attn_implementation="eager"
)

vit_v2 = AutoModel.from_pretrained(
    VIT_MODEL,
    attn_implementation="eager"
)

# Freeze all BERT layers
for param in bert_v2.parameters():
    param.requires_grad = False

# Unfreeze last four BERT layers
for layer in bert_v2.encoder.layer[-4:]:
    for param in layer.parameters():
        param.requires_grad = True

# Freeze ViT
for param in vit_v2.parameters():
    param.requires_grad = False

bert_v2 = bert_v2.to(device)
vit_v2 = vit_v2.to(device)

print("BERT V2 and ViT V2 loaded.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.weight   | UNEXPECTED | 
classifier.bias     | UNEXPECTED | 
pooler.dense.weight | MISSING    | 
pooler.dense.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT V2 and ViT V2 loaded.


### Building Early Fusion V2

This model combines text features from fine-tuned BERT and image features from frozen ViT. The last four BERT layers are trainable, allowing the model to adapt to ScienceQA. The extracted text and image features are concatenated and passed through a classifier to predict the correct answer.

In [24]:
class EarlyFusionModelV2(nn.Module):
    def __init__(self, bert, vit):
        super().__init__()

        self.bert = bert
        self.vit = vit

        # Fusion classifier
        self.fusion = nn.Sequential(
            nn.Linear(768 + 768, 512),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(512, 1)
        )

    def train(self, mode=True):
        super().train(mode)

        # Keep frozen ViT in evaluation mode
        self.vit.eval()

        return self

    def forward(self, input_ids, attention_mask, pixel_values, option_mask):

        batch_size, num_choices, seq_length = input_ids.shape

        input_ids = input_ids.view(-1, seq_length)
        attention_mask = attention_mask.view(-1, seq_length)

        valid_indices = option_mask.view(-1).nonzero(as_tuple=True)[0]

        # Extract text features
        text_output = self.bert(
            input_ids=input_ids[valid_indices],
            attention_mask=attention_mask[valid_indices]
        )

        text_features = text_output.last_hidden_state[:, 0, :]

        # Extract image features
        with torch.no_grad():
            image_output = self.vit(pixel_values=pixel_values)
            image_features = image_output.last_hidden_state[:, 0, :]

        # Repeat image features for each valid answer option
        image_features = image_features.repeat_interleave(
            num_choices, dim=0
        )[valid_indices]

        # Combine text and image features
        combined_features = torch.cat(
            (text_features, image_features), dim=1
        )

        valid_scores = self.fusion(combined_features).squeeze(-1)

        scores = torch.full(
            (batch_size * num_choices,),
            float("-inf"),
            device=input_ids.device
        )

        scores[valid_indices] = valid_scores

        return scores.view(batch_size, num_choices)


early_fusion_v2 = EarlyFusionModelV2(
    bert_v2, vit_v2
).to(device)

print("Early Fusion V2 created successfully.")

Early Fusion V2 created successfully.


### Configuring the Optimizer

This cell configures the AdamW optimizer with separate learning rates. A smaller learning rate is used for BERT because it is pretrained, while a higher learning rate is used for the newly created fusion classifier.

In [25]:
criterion_v2 = nn.CrossEntropyLoss()

optimizer_v2 = torch.optim.AdamW(
    [
        {
            "params": [
                p for p in early_fusion_v2.bert.parameters()
                if p.requires_grad
            ],
            "lr": 2e-5
        },
        {
            "params": early_fusion_v2.fusion.parameters(),
            "lr": 1e-3
        }
    ],
    weight_decay=0.01
)

print("V2 optimizer configured.")

V2 optimizer configured.


I define separate training and evaluation functions for Early Fusion V2. The model learns through the fusion classifier and the final four BERT layers, while the ViT image encoder remains frozen.

In [26]:
from tqdm.auto import tqdm
import torch
import torch.nn as nn

def train_one_epoch_v2(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    progress_bar = tqdm(loader, desc="Training V2")

    for batch in progress_bar:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values = batch["pixel_values"].to(device)
        option_mask = batch["option_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids,
            attention_mask,
            pixel_values,
            option_mask
        )

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        predictions = outputs.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

        progress_bar.set_postfix(loss=loss.item())

    return total_loss / len(loader), correct / total


def evaluate_v2(model, loader, criterion):
    model.eval()

    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            pixel_values = batch["pixel_values"].to(device)
            option_mask = batch["option_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(
                input_ids,
                attention_mask,
                pixel_values,
                option_mask
            )

            loss = criterion(outputs, labels)

            total_loss += loss.item()
            predictions = outputs.argmax(dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    return total_loss / len(loader), correct / total

In [27]:
import copy

EPOCHS_V2 = 20
PATIENCE_V2 = 4

history_v2 = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

best_val_loss_v2 = float("inf")
best_epoch_v2 = 0
patience_counter_v2 = 0
best_model_weights_v2 = None

for epoch in range(EPOCHS_V2):
    print(f"\nEpoch {epoch + 1}/{EPOCHS_V2}")

    train_loss, train_accuracy = train_one_epoch_v2(
        early_fusion_v2,
        train_loader,
        optimizer_v2,
        criterion_v2
    )

    val_loss, val_accuracy = evaluate_v2(
        early_fusion_v2,
        val_loader,
        criterion_v2
    )

    history_v2["train_loss"].append(train_loss)
    history_v2["train_accuracy"].append(train_accuracy)
    history_v2["val_loss"].append(val_loss)
    history_v2["val_accuracy"].append(val_accuracy)

    print(f"Train Loss: {train_loss:.4f} | Train Accuracy: {train_accuracy:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val Accuracy:   {val_accuracy:.4f}")

    if val_loss < best_val_loss_v2:
        best_val_loss_v2 = val_loss
        best_epoch_v2 = epoch + 1
        patience_counter_v2 = 0
        best_model_weights_v2 = copy.deepcopy(early_fusion_v2.state_dict())

        print("Best model updated.")

    else:
        patience_counter_v2 += 1
        print(f"No improvement: {patience_counter_v2}/{PATIENCE_V2}")

    if patience_counter_v2 >= PATIENCE_V2:
        print("Early stopping triggered.")
        break

early_fusion_v2.load_state_dict(best_model_weights_v2)

print("\nV2 Training completed.")
print(f"Best epoch: {best_epoch_v2}")
print(f"Best validation loss: {best_val_loss_v2:.4f}")


Epoch 1/20


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 1.0971 | Train Accuracy: 0.4292
Val Loss:   0.7442 | Val Accuracy:   0.6076
Best model updated.

Epoch 2/20


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.8500 | Train Accuracy: 0.5635
Val Loss:   0.6719 | Val Accuracy:   0.6560
Best model updated.

Epoch 3/20


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.7436 | Train Accuracy: 0.6229
Val Loss:   0.6702 | Val Accuracy:   0.6530
Best model updated.

Epoch 4/20


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

Train Loss: 0.7050 | Train Accuracy: 0.6425
Val Loss:   0.6512 | Val Accuracy:   0.6560
Best model updated.

Epoch 5/20


Training V2:   0%|          | 0/442 [00:00<?, ?it/s]

KeyboardInterrupt: 